# 🔄 GitHub Modelsを使った基本的なエージェントワークフロー（Python）

## 📋 ワークフローオーケストレーションチュートリアル

このノートブックでは、Microsoft Agent Frameworkの強力な<strong>Workflow Builder</strong>機能を紹介します。複雑なビジネスプロセスを処理し、複数のAI操作をシームレスに調整できる洗練された多段階エージェントワークフローの作成方法を学びましょう。

## 🎯 学習目標

### 🏗️ <strong>ワークフローアーキテクチャ</strong>
- **Workflow Builder**：複雑な多段階プロセスの設計とオーケストレーション
- <strong>イベント駆動実行</strong>：ワークフローのイベントと状態遷移の処理
- <strong>ビジュアルワークフロー設計</strong>：ワークフロー構造の作成と可視化
- **GitHub Models統合**：ワークフローコンテキスト内でのAIモデル活用

### 🔄 <strong>プロセスオーケストレーション</strong>
- <strong>順次操作</strong>：複数のエージェントタスクを論理的に連結
- <strong>条件ロジック</strong>：意思決定ポイントと分岐ワークフローの実装
- <strong>エラー処理</strong>：堅牢なエラー回復とワークフローのレジリエンス
- <strong>状態管理</strong>：ワークフロー実行状態の追跡と管理

### 📊 <strong>エンタープライズワークフローパターン</strong>
- <strong>ビジネスプロセス自動化</strong>：複雑な組織のワークフローの自動化
- <strong>マルチエージェント調整</strong>：複数の専門エージェントの調整
- <strong>スケーラブルな実行</strong>：エンタープライズレベルの業務対応設計
- <strong>モニタリングと可観測性</strong>：ワークフロー性能と結果の追跡

## ⚙️ 前提条件とセットアップ

### 📦 <strong>必要な依存関係</strong>

ワークフロー機能付きのAgent Frameworkをインストール：

```bash
pip install agent-framework-core -U
```

### 🔑 **GitHub Modelsの設定**

**環境セットアップ（.envファイル）：**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 <strong>エンタープライズユースケース</strong>

**ビジネスプロセス例：**
- <strong>顧客オンボーディング</strong>：複数ステップの検証とセットアップワークフロー
- <strong>コンテンツパイプライン</strong>：自動コンテンツ作成、レビュー、公開
- <strong>データ処理</strong>：AI駆動の変換を含むETLワークフロー
- <strong>品質保証</strong>：自動テストと検証プロセス

**ワークフローの利点：**
- 🎯 <strong>信頼性</strong>：エラー回復を備えた決定的実行
- 📈 <strong>スケーラビリティ</strong>：高ボリュームプロセス自動化対応
- 🔍 <strong>可観測性</strong>：完全な監査証跡とモニタリング
- 🔧 <strong>保守性</strong>：ビジュアル設計とモジュールコンポーネント

## 🎨 ワークフロー設計パターン

### 基本的なワークフロー構造
```mermaid
graph TD
    A[開始] --> B[エージェントタスク1]
    B --> C{決定点}
    C -->|成功| D[エージェントタスク2]
    C -->|失敗| E[エラーハンドラー]
    D --> F[終了]
    E --> F
```

**主なコンポーネント：**
- **WorkflowBuilder**：メインのオーケストレーションエンジン
- **WorkflowEvent**：イベントの処理と通信
- **WorkflowViz**：ワークフローの可視化表現とデバッグ

さあ、あなたの最初のインテリジェントなワークフローを作りましょう！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
